# Ligand preparation for AutoDock Vina

This notebook converts ligands from SDF and SMILES formats into PDBQT format for docking with AutoDock Vina

Pipeline:
- Read input molecules (SDF and SMI)
- Add hydrogens and generate 3D coordinates if needed
- Prepare molecules using Meeko
- Export to PDBQT format

## Imports

In [1]:
from pathlib import Path

from rdkit import Chem
from rdkit.Chem import AllChem

from meeko import MoleculePreparation, PDBQTWriterLegacy

## Configuration

In [2]:
INPUT_SDF = Path("input/example.sdf")
INPUT_SMI = Path("input/example.smi")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Functions

In [3]:
def prepare_molecule_rdkit(mol):
    mol = Chem.Mol(mol)
    mol = Chem.AddHs(mol)

    is_3d = False
    if mol.GetNumConformers() > 0:
        is_3d = mol.GetConformer().Is3D()

    if not is_3d:
        params = AllChem.ETKDGv3()
        params.randomSeed = 42

        status = AllChem.EmbedMolecule(mol, params)
        if status != 0:
            raise ValueError("Failed to generate 3D conformer")

        AllChem.UFFOptimizeMolecule(mol)

    return mol


def write_pdbqt(mol, output_path, flexible_macrocycles=True):
    """Prepare molecule with Meeko and write to PDBQT file.
    
    Aliphatic rings with 6+ atoms are treated as flexible by default.
    """
    preparator = MoleculePreparation(
        rigid_macrocycles=not flexible_macrocycles,
        min_ring_size=6,
    )
    setups = preparator.prepare(mol)

    if not setups:
        raise ValueError("Meeko failed to prepare molecule")

    pdbqt_string, is_ok, error_msg = PDBQTWriterLegacy.write_string(setups[0])

    if not is_ok:
        raise ValueError(error_msg)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(pdbqt_string)

## Load data

Quick exploratory look at the input files before processing

In [4]:
# SDF overview
supplier = Chem.SDMolSupplier(str(INPUT_SDF))
print("Molecules in SDF:", len(supplier))

for i, mol in enumerate(supplier):
    if mol is None:
        print(f"  [{i}] parse error")
        continue

    name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"ligand_{i}"
    conformers = mol.GetNumConformers()
    is_3d = mol.GetConformer().Is3D() if conformers > 0 else False
    print(f"  [{i}] {name} | atoms: {mol.GetNumAtoms()} | conformers: {conformers} | 3D: {is_3d}")

Molecules in SDF: 30
  [0] CHEMBL3957375 | atoms: 31 | conformers: 1 | 3D: False
  [1] CHEMBL3104445 | atoms: 35 | conformers: 1 | 3D: False
  [2] CHEMBL3901969 | atoms: 34 | conformers: 1 | 3D: False
  [3] CHEMBL4800145 | atoms: 32 | conformers: 1 | 3D: False
  [4] CHEMBL3957572 | atoms: 32 | conformers: 1 | 3D: False
  [5] CHEMBL3934158 | atoms: 36 | conformers: 1 | 3D: False
  [6] CHEMBL3461502 | atoms: 23 | conformers: 1 | 3D: False
  [7] CHEMBL3982666 | atoms: 39 | conformers: 1 | 3D: False
  [8] CHEMBL4109555 | atoms: 31 | conformers: 1 | 3D: False
  [9] CHEMBL494995 | atoms: 27 | conformers: 1 | 3D: False
  [10] CHEMBL3699816 | atoms: 30 | conformers: 1 | 3D: False
  [11] CHEMBL1346176 | atoms: 28 | conformers: 1 | 3D: False
  [12] CHEMBL104844 | atoms: 26 | conformers: 1 | 3D: False
  [13] CHEMBL3976526 | atoms: 29 | conformers: 1 | 3D: False
  [14] CHEMBL1386741 | atoms: 20 | conformers: 1 | 3D: False
  [15] CHEMBL1644677 | atoms: 29 | conformers: 1 | 3D: False
  [16] CHEMBL45

In [5]:
# SMI overview — first 5 lines
with open(INPUT_SMI, "r") as f:
    for _ in range(5):
        print(f.readline().strip())

smiles name
CC(=O)N1C[C@@H]2OCC(=O)N(Cc3ccc(F)cc3)[C@@H]2C1 CHEMBL4959907
Cc1nn(C)cc1-c1cc(C(N)=O)nc2cc(CN3C[C@@H](C)O[C@@H](C(F)(F)F)C3)ccc12 CHEMBL4108673
Cc1nsc(-c2cc(C(N)=O)nc3cc(CN4C[C@@H](C)O[C@@H](C(F)(F)F)C4)ccc23)n1 CHEMBL4107154
Fc1ccc(CN2CCOC(Cn3cncn3)C2)c2ncccc12 CHEMBL3460453


## Process SDF ligands

In [6]:
supplier = Chem.SDMolSupplier(str(INPUT_SDF), removeHs=False)

sdf_success = 0
sdf_failed = 0

for i, mol in enumerate(supplier, start=1):
    if mol is None:
        print(f"[{i}] parse error")
        sdf_failed += 1
        continue

    name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"ligand_{i}"

    try:
        prepared_mol = prepare_molecule_rdkit(mol)
        write_pdbqt(prepared_mol, OUTPUT_DIR / f"{name}.pdbqt")
        sdf_success += 1
        print(f"Saved: {name}.pdbqt")

    except Exception as e:
        sdf_failed += 1
        print(f"Error for {name}: {e}")

print("---")
print("SDF success:", sdf_success)
print("SDF failed: ", sdf_failed)

Saved: CHEMBL3957375.pdbqt
Saved: CHEMBL3104445.pdbqt
Saved: CHEMBL3901969.pdbqt
Saved: CHEMBL4800145.pdbqt
Saved: CHEMBL3957572.pdbqt
Saved: CHEMBL3934158.pdbqt
Saved: CHEMBL3461502.pdbqt
Saved: CHEMBL3982666.pdbqt
Saved: CHEMBL4109555.pdbqt
Saved: CHEMBL494995.pdbqt
Saved: CHEMBL3699816.pdbqt
Saved: CHEMBL1346176.pdbqt
Saved: CHEMBL104844.pdbqt
Saved: CHEMBL3976526.pdbqt
Saved: CHEMBL1386741.pdbqt
Saved: CHEMBL1644677.pdbqt
Saved: CHEMBL4565308.pdbqt
Saved: CHEMBL4556129.pdbqt
Saved: CHEMBL3922226.pdbqt
Saved: CHEMBL352017.pdbqt
Saved: CHEMBL1308413.pdbqt
Saved: CHEMBL348133.pdbqt
Saved: CHEMBL3646377.pdbqt
Saved: CHEMBL3649287.pdbqt
Saved: CHEMBL3646372.pdbqt
Saved: CHEMBL3461559.pdbqt
Saved: CHEMBL3986453.pdbqt
Saved: CHEMBL3946996.pdbqt
Saved: CHEMBL3699817.pdbqt
Saved: CHEMBL3894759.pdbqt
---
SDF success: 30
SDF failed:  0


## Process SMI ligands

SMILES input does not contain 3D coordinates, so 3D conformers are generated using RDKit before conversion to PDBQT

In [7]:
smi_success = 0
smi_failed = 0

with open(INPUT_SMI, "r") as f:
    next(f)  # skip header: "smiles name"

    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        parts = line.split()
        smiles = parts[0]
        name = parts[1] if len(parts) > 1 else f"ligand_{i}"

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Invalid SMILES [{i}]: {smiles}")
            smi_failed += 1
            continue

        try:
            prepared_mol = prepare_molecule_rdkit(mol)
            write_pdbqt(prepared_mol, OUTPUT_DIR / f"smi_{name}.pdbqt")
            smi_success += 1

        except Exception as e:
            smi_failed += 1
            print(f"Error for {name}: {e}")

print("---")
print("SMI success:", smi_success)
print("SMI failed: ", smi_failed)

[RDKit] ERROR:[17:37:40] UFFTYPER: Unrecognized charge state for atom: 27
[RDKit] ERROR:[17:37:40] UFFTYPER: Unrecognized charge state for atom: 27


---
SMI success: 30
SMI failed:  0


## Summary

In [10]:
total_success = sdf_success + smi_success
total_failed = sdf_failed + smi_failed

print("=" * 30)
print(f"SDF  — success: {sdf_success}, failed: {sdf_failed}")
print(f"SMI  — success: {smi_success}, failed: {smi_failed}")
print("-" * 30)
print(f"Total — success: {total_success}, failed: {total_failed}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

SDF  — success: 30, failed: 0
SMI  — success: 30, failed: 0
------------------------------
Total — success: 60, failed: 0
Output directory: C:\Users\verna\Documents\ligand-preparation-vina\output
